<a href="https://colab.research.google.com/github/siinwook/Deep-Learning-from-Scratch/blob/main/layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
class Relu:
  def __init__(self):
    self.mask = None # is zero or negative

  def forward(self,x):
    self.mask = (x<=0)
    out = x.copy()
    out[self.mask]=0

    return out

  def backward(self,dout):
    dout[self.mask] = 0
    dx = dout

    return dx

In [ ]:
class Sigmoid:
  def _init__(self):
    self.out = None

  def forward(self, x):
    out = 1 / (1+np.exp(-x))
    self.out = out

    return out

  def backward(self, dout):
    dx = dout * self.out * (1-self.out) #d/dx[sigmoid(x)] = sigmoid(x) * (1 - sigmoid(x))

    return dx

In [ ]:
class Affine:
  def __init__(self,W,b):
    self.W = W
    self.b = b
    self.x = None
    self.dW = None
    self.db = None

    self.original_x_shape = None #for img(4-dim)

  def forward(self,x):
    self.original_x_shape = x.shape
    x = x.reshape(x.shape[0],-1)
    self.x = x

    out = np.dot(x,self.W) + self.b

    return out

  def backward(self, dout):
    dx = np.dot(dout, self.W.T) #dx = dout dot W.T
    self.dW = np.dot(self.x.T, dout) #dW = x.T dot dout
    self.db = np.sum(dout,axis=0) #db = sum(dout)

    dx = dx.reshape(*self.original_x_shape)
    return dx

In [ ]:
def softmax(x):
  exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
  return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy_error(y,t):
  if y.ndim==1:
    t=t.reshape(1,t.size)
    y=y.reshape(1,y.size)

  if t.size==y.size:
    t=t.argmax(axis=1)

  batch_size = y.shape[0]
  delta = 1e-7
  return -np.sum(np.log(y[np.arange(batch_size), t] + delta)) / batch_size

class SoftmaxWithLoss:
  def __init__(self):
    self.loss = None
    self.y = None
    self.t = None

  def forward(self,x,t):
    self.t = t
    self.y = softmax(x)
    self.loss = cross_entropy_error(self.y, self.t)

    return self.loss

  def backward(self, dout):
    batch_size = self.t.shape[0]

    dx = (self.y - self.t) / batch_size

    return dx

In [ ]:
class Dropout:
  def __init__(self,dropout_ratio = 0.5):
    self.dropout_ratio = dropout_ratio
    self.mask = None

  def forward(self,x, train_flg=True):
    if train_flg:
      self.mask = np.random.rand(*x.shape) > self.dropout_ratio
      return x * self.mask

    else:
      return x * (1.0 - self.dropout_ratio)

  def backward(self, dout):
    return dout * self.mask

In [ ]:
class BatchNormalization:
  def __init__(self,gamma, beta, train_flg=True, momentum=0.9, running_mean=None, running_var = None):
    self.input_shape = None
    self.gamma = gamma
    self.beta = beta
    self.momentum = momentum

    self.running_mean = running_mean
    self.running_var = running_var

    self.batch_size = None
    self.xc = None
    self.xn = None
    self.std = None
    self.dgamma = None
    self.dbeta = None

  def forward(self, x, train_flg):
    self.input_shape = x.shape
    if x.ndim!=2: #for CNN
      N,C,H,W = x.shape
      x = x.reshape(N,-1)

    out = self.__forward(x,train_flg)
    return out.reshape(*self.input_shape)

  def __forward(self, x, train_flg=True):
    if self.running_mean is None:
      N,D = x.shape
      self.running_mean = np.zeros_like(D)
      self.running_var = np.zeros_like(D)

    if train_flg:
      mu = x.mean(axis=0)
      xc = x - mu
      var = np.mean(xc**2,axis=0)
      std = np.sqrt(var+1e-7)
      xn = xc / std

      self.batch_size = x.shape[0]
      self.xc = xc
      self.xn = xn
      self.std = std
      self.running_mean = self.momentum * self.running_mean + (1-self.momentum) * mu
      self.running_var = self.momentum * self.running_var + (1-self.momentum) * var
    else:
      xc = x - self.running_mean
      xn = xc / np.sqrt(self.running_var + 1e-7)

    out = self.gamma * xn + self.beta
    return out

  def backward(self, dout):
    if dout.ndim != 2: #for CNN
      N,C,H,W = dout.shape
      dout = dout.reshape(N,-1)

    dx = self.__backward(dout)

    dx = dx.reshape(*self.input_shape)
    return dx

  def __backward(self, dout):
    dbeta = np.sum(dout,axis=0)
    dgamma = np.sum(dout * self.xn, axis=0)
    dxn = self.gamma * dout
    dxc = dxn / self.std
    dstd = -np.sum((dxn * self.xc) / self.std**2, axis=0)
    dvar = 0.5 * dstd / self.std
    dxc += (2.0/self.batch_size) * self.xc * dvar
    dmu = np.sum(dxc, axis=0)
    dx = dxc - dmu / self.batch_size

    self.dbeta = dbeta
    self.dgamma = dgamma
    return dx

In [ ]:
class Convolution:
  def __init__(self, W,b,stride=1,pad=0):
    self.W = W #(FN,C,FH,FW)
    self.b = b #(FN)
    self.stride = stride
    self.pad = pad

    self.dW=None
    self.db=None

    #for backward
    self.x=None
    self.col=None
    self.col_W=None

  def forward(self,x): #(N,C,H,W) -> (N,FN,outH,outW)
    N,C,W,H = x.shape
    FN,C,FH,FW = self.W.shape

    out_h = int((H + 2*self.pad - FH)/ self.stride + 1)
    out_w = int((W + 2*self.pad - FW)/ self.stride + 1)

    col = im2col(x,FH,FW,self.stride,self.pad) #(N,C,H,W) -> (N*outH*outW,C*FH*FW)
    col_W = self.W.reshape(FN,-1).T #(FN,C*FH*FW) -> (C*FH*FW,FN)

    out = np.dot(col,col_W) + self.b #(N*outH*outW,C*FH*FW) dot (C*FH*FW,FN) -> (N*outH*outW,FN)
    out = out.reshape(N,out_h,out_w,-1).transpose(0,3,1,2) #(N*outH*outW,FN) -> (N,outH,outW,FN) -> (N,FN,outH,outW)

    self.x = x
    self.col = col
    self.col_W = col_W

    return out #(N,FN,outH,outW)

  def backward(self, dout): #(N,FN,outH,outW) -> (N,C,H,W)
    FN,C,FW,FH = self.W.shape
    dout = dout.transpose(0,2,3,1).reshape(-1,FN) #(N,FN,outH,outW) -> (N,outH,outW,FN) -> (N*outH*outW,FN)

    self.db = np.sum(dout, axis=0)
    self.dW = np.dot(self.col.T, dout) #(C*FH*FW,N*outH*outW) dot (N*H*W,FN) -> (C*FH*FW,FN)
    self.dW = self.dW.transpose(1,0).reshape(FN,C,FH,FW) #(C*FH*FW,FN) -> (FN,C*FH*FW) -> (FN,C,FH,FW)

    dcol = np.dot(dout, self.col_W.T) #(N*outH*outW,FN) dot (FN,C*FH*FW) -> (N*outH*outW,C*FH*FW)
    dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad) #(N*outH*outW,C*FH*FW) -> (N,C,H,W)

    return dx #(N,C,H,W)

In [ ]:
class Pooling:
  def __init__(self,pool_h,pool_w,stride=1,pad=0):
    self.pool_h = pool_h
    self.pool_w = pool_w
    self.stride = stride
    self.pad = pad

    self.x = None
    self.arg_max = None
  def forward(self, x): #(N,C,H,W) -> (N,C,outH,outW)
    N,C,H,W = x.shape
    out_h = int(1+ (H - self.pool_h) / self.stride)
    out_w = int(1+ (W - self.pool_w) / self.stride)

    col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad) #(N,C,H,W) -> (N*outH*outW,C*FH*FW)
    col = col.reshape(-1, self.pool_h * self.pool_w) #(N*outH*outW,C*FH*FW) -> (N*outH*outW*C,FH*FW)

    self.arg_max = np.argmax(col, axis=1) #(N*outH*outW*C,)
    out = np.max(col, axis=1) #(N*outH*outW*C,FH*FW) -> (N*outH*outW*C,)

    out = out.reshape(N, out_h, out_w, C).transpose(0,3,1,2) #(N*outH*outW*C,) -> (N,outH,outW,C) -> (N,C,outH,outW)

    self.x = x
    return out #(N,C,outH,outW)

  def backward(self, dout): #(N,C,outH,outW) -> (N,C,H,W)
    dout = dout.transpose(0,2,3,1) #(N,C,outH,outW) -> (N,outH,outW,C)

    pool_size = self.pool_h * self.pool_w
    dmax = np.zeros((dout.size, pool_size)) #(N*outH*outW*C,FH*FW)
    dmax[np.arange(self.arg_max.size),self.arg_max.flatten()] = dout.flatten() #put dout into dmax[:,index of max element]
    dmax = dmax.reshape(dout.shape + (pool_size,)) #(N*outH*outW*C,FH*FW)

    dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2], -1) #(N*outH*outW*C,FH*FW) -> (N*outH*outW,C*FH*FW)
    dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad) #(N*outH*outW,C*FH*FW) -> (N,C,H,W)

    return dx #(N,C,H,W)